# 깨끗한 표면과 마스킹 전처리 사진의 GAN 예비 생성

기존 DCGAN을 처음부터 학습하여 네 조건에서 각각 50장씩, 총 200장을 생성합니다. ① 깨끗한 정상 표면, ② 깨끗한 언더필 표면, ③ 마스킹한 정상 부분 표면, ④ 마스킹한 언더필 부분 표면입니다. 실제 입력은 조건당 50장입니다. **분류기는 학습하지 않습니다.**

깨끗한 사진과 마스킹 사진은 원본 구성이 다르므로 결과 차이를 마스킹 유무 하나의 인과효과로 볼 수 없습니다. 마스킹 조건은 추가 활용 사진의 회색 마스킹을 포함합니다. 자동 추정 마스크는 정답이 아니며, 원본 라벨은 출력 조건으로서 각 사진에 명확한 결함이 보임을 보장하지 않습니다.

이 시각 검토용 사진 목록은 전체 촬영 구간에서 추출되었으므로, 이번 GAN이 생성한 이미지를 추후 분류기 학습에 사용할 경우 평가 프레임이나 인접 프레임이 겹칠 수 있습니다. 추후 성능 실험은 학습 구간만으로 GAN을 재학습하거나, 독립된 새 출력 데이터로 평가해야 합니다.

런타임 유형에서 G4를 선택하세요. 각 GAN은 독립적으로 학습하며 학습 데이터와 생성 이미지는 256×256 RGB입니다. 동일한 DCGAN, 시드 42, 배치 8, 생성기·판별기 1,500회 업데이트입니다. 이 값은 이전 예비 시험과 비교하기 위한 설정이지 최적 조건이 아닙니다. 네 모델의 생성 결과와 로그를 Drive에 저장한 후 G4 런타임을 자동 반납합니다. 강제 종료나 연결 문제 시 자동 반납이 보장되지 않으므로 수동 확인이 필요합니다.

In [ ]:
from google.colab import runtime, drive
import traceback, sys, os, json, hashlib, time, subprocess, zipfile
from pathlib import Path, PurePosixPath

mounted = False
run_root = None
try:
    drive.mount('/content/drive')
    mounted = True
    import torch
    from PIL import Image, ImageOps, ImageDraw
    from IPython.display import display

    drive_root = Path('/content/drive/MyDrive/산공종설')
    data_zip = drive_root / 'gan_four_conditions_20261101_data.zip'
    code_zip = drive_root / 'gan_four_conditions_20261101_code.zip'
    run_root = drive_root / 'results/gan_four_conditions_20261101_run01'
    prepared = run_root / 'prepared'
    output = run_root / 'gan'
    review = run_root / 'review'
    logs = run_root / 'logs'
    code_dir = Path('/content/gan_four_conditions_code')
    conditions = ('clean_normal', 'clean_underfill', 'partial_normal', 'partial_underfill')
    seed, steps, batch = 42, 1500, 8
    for path in (data_zip, code_zip):
        if not path.is_file(): raise FileNotFoundError(f'필요한 Drive 파일 없음: {path}')
    if not torch.cuda.is_available(): raise RuntimeError('G4 GPU를 선택하세요.')
    gpu = torch.cuda.get_device_name(0)
    print('사용 GPU:', gpu, flush=True)
    if 'RTX PRO 6000' not in gpu.upper(): raise RuntimeError(f'G4가 아닙니다: {gpu}')

    def sha(path):
        digest = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda: stream.read(4 * 1024 * 1024), b''): digest.update(block)
        return digest.hexdigest()
    data_sha, code_sha = sha(data_zip), sha(code_zip)
    if data_sha != 'c8db7010769b7c950f3b0400ddbc4be9c88e2e99fbe90000f290d721d0eb68ac' or code_sha != '55748570a236233a25d1f459453edebf1c7bf2311ecd37c002eac55fb3a81b32':
        raise ValueError('노트북에 기록된 입력 또는 코드 ZIP과 다릅니다.')
    settings = {'seed': seed, 'steps': steps, 'batch': batch, 'gpu': gpu,
                'data_sha256': data_sha, 'code_sha256': code_sha,
                'conditions': list(conditions), 'actual_per_condition': 50,
                'generated_per_condition': 50, 'classifier_training': False,
                'warning': '전체 촬영 구간에서 뽑은 검토 사진. 추후 성능 실험은 학습 구간만으로 GAN 재학습 또는 독립 새 출력 필요.'}
    run_root.mkdir(parents=True, exist_ok=True)
    review.mkdir(exist_ok=True); logs.mkdir(exist_ok=True)
    lock = run_root / 'run_settings.json'
    if lock.exists():
        old = json.loads(lock.read_text(encoding='utf-8'))
        for key in ('seed','steps','batch','data_sha256','code_sha256','conditions'):
            if old[key] != settings[key]: raise ValueError('기존 실험 설정이 달라 중단합니다.')
    else:
        lock.write_text(json.dumps(settings, ensure_ascii=False, indent=2), encoding='utf-8')
    readme = run_root / 'README_실험설명.md'
    readme.write_text('# 네 조건의 GAN 예비 생성\n\n'
'깨끗한 정상·언더필과 마스킹한 부분 정상·언더필 사진을 각각 실제 50장씩 사용한다. '
'조건마다 별도의 DCGAN을 처음부터 학습해 이미지 50장씩 생성한다. '
'DCGAN 구조·시드 42·배치 8·1500회 업데이트는 이전 예비실험과 비교하기 위한 설정이며 최적이라는 근거는 없다.\n\n'
'사진 원본과 선별 경로는 prepared/input_manifest.csv와 selection_settings.json에 기록한다. '
'고정 1000×1000 원본 크롭을 256×256으로 줄였고, 마스킹 그룹은 출력물 바깥을 회색으로 가린다. '
'깨끗한 그룹과 마스킹 그룹은 사진 원본이 달라 결과 차이를 마스킹 자체의 효과라고 단정할 수 없다.\n\n'
'이번 입력은 기존 평가 구간을 포함한 전체 촬영 구간에서 추출한 시각 검토용이다. '
'추후 성능 실험은 학습 구간만으로 GAN 재학습 또는 독립 새 출력 필요. '
'각 클래스는 기존 출력 1회에서 얻은 사진이므로 출력 간 일반화는 확인되지 않는다. '
'생성 사진의 실제 결함 재현 여부 및 분류기 성능은 이번 실험에서 확인하지 않는다.\n\n'
'조건별 체크포인트는 gan 폴더에 보관하고 결과 ZIP에는 포함하지 않는다.\n', encoding='utf-8')

    code_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(code_zip) as archive:
        if archive.namelist() != ['gan_model.py']: raise ValueError('코드 ZIP 구성이 다릅니다.')
        (code_dir / 'gan_model.py').write_bytes(archive.read('gan_model.py'))
    marker = prepared / '.data_complete.json'
    if marker.exists():
        if json.loads(marker.read_text(encoding='utf-8')).get('sha256') != data_sha:
            raise ValueError('기존 입력의 지문이 달라 중단합니다.')
    else:
        if prepared.exists() and any(prepared.iterdir()):
            raise RuntimeError('기존 데이터 준비가 중단됐습니다. 새 출력 폴더를 사용하세요.')
        prepared.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(data_zip) as archive:
            for entry in archive.infolist():
                parts = PurePosixPath(entry.filename).parts
                if not parts or entry.filename.startswith('/') or any(x in ('..','') for x in parts) or '\\' in entry.filename:
                    raise ValueError(f'안전하지 않은 입력 경로: {entry.filename}')
                dest = prepared.joinpath(*parts)
                if entry.is_dir():
                    dest.mkdir(parents=True, exist_ok=True)
                else:
                    dest.parent.mkdir(parents=True, exist_ok=True)
                    dest.write_bytes(archive.read(entry))
        marker.write_text(json.dumps({'sha256':data_sha}), encoding='utf-8')

    def run_logged(command, log_file):
        print('실행:', ' '.join(str(x) for x in command), flush=True)
        with log_file.open('a', encoding='utf-8') as log:
            log.write('\n--- ' + time.strftime('%Y-%m-%d %H:%M:%S') + ' ---\n'); log.flush()
            proc = subprocess.Popen([str(x) for x in command], stdout=subprocess.PIPE,
                                    stderr=subprocess.STDOUT, text=True, encoding='utf-8',
                                    errors='replace', bufsize=1)
            try:
                for line in proc.stdout:
                    log.write(line); log.flush(); print(line, end='', flush=True)
                result = proc.wait()
            except BaseException:
                proc.terminate()
                raise
            if result: raise RuntimeError(f'GAN 실패: {log_file}, 종료 코드 {result}')

    def sheet(paths, dest):
        width, height, cap, cols = 180, 180, 24, 10
        board = Image.new('RGB', (width * cols, (height + cap) * ((len(paths)+cols-1)//cols)), 'white')
        draw = ImageDraw.Draw(board)
        for i, path in enumerate(paths):
            x, y = (i % cols)*width, (i // cols)*(height+cap)
            with Image.open(path) as image:
                board.paste(ImageOps.fit(image.convert('RGB'), (width,height), method=Image.Resampling.LANCZOS), (x,y))
            draw.text((x+3,y+height+3), path.stem[:23], fill='black')
        board.save(dest, quality=92)

    summary = {}
    for no, condition in enumerate(conditions, 1):
        data_dir = prepared / 'data' / condition
        actual = sorted(data_dir.glob('*.png'))
        if len(actual) != 50: raise RuntimeError(f'{condition} 실사 {len(actual)}장: 50장 필요')
        for p in actual:
            with Image.open(p) as im:
                if im.mode != 'RGB' or im.size != (256,256): raise ValueError(f'입력 RGB 256×256 오류: {p}')
        print(f'[{no}/4] {condition}: 실사 50장으로 GAN 학습 시작', flush=True)
        gan = output / condition
        run_logged([sys.executable, code_dir/'gan_model.py', '--data', data_dir,
                    '--out', gan, '--steps', steps, '--seed', seed, '--num-generate', 50,
                    '--device','cuda','--batch',batch,'--save-every',100,'--log-every',25],
                   logs/f'{condition}.log')
        progress = json.loads((gan/'progress.json').read_text(encoding='utf-8'))
        manifest = json.loads((gan/'training_manifest.json').read_text(encoding='utf-8'))
        generated = sorted((gan/'generated').glob('*.png'))
        expected_names = [f'generated_{i:03d}.png' for i in range(1,51)]
        if progress.get('status') != 'completed' or progress.get('step') != steps or [p.name for p in generated] != expected_names:
            raise RuntimeError(f'{condition}: 학습 완료/출력 50장 확인 실패')
        if {x['filename']:x['sha256'] for x in manifest['images']} != {p.name:sha(p) for p in actual}:
            raise RuntimeError(f'{condition}: 학습 사진 지문이 다릅니다')
        sheet(actual, review/f'{condition}_real_50.jpg')
        sheet(generated, review/f'{condition}_generated_50.jpg')
        summary[condition] = {'actual':len(actual), 'generated':len(generated), 'status':'completed',
                              'actual_preview':str(review/f'{condition}_real_50.jpg'),
                              'generated_preview':str(review/f'{condition}_generated_50.jpg')}
        print(f'[{no}/4] {condition}: 50장 생성 완료', flush=True)

    result = run_root/'gan_four_conditions_20261101_run01_results.zip'
    tmp = result.with_suffix('.zip.part')
    if result.exists():
        with zipfile.ZipFile(result) as archive:
            if archive.testzip() is not None: raise RuntimeError('기존 결과 ZIP 손상')
        print('기존 검증된 ZIP 유지:', result, flush=True)
    else:
        with zipfile.ZipFile(tmp, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=4) as archive:
            for root in (prepared, output, review, logs):
                for path in root.rglob('*'):
                    if not path.is_file() or path.suffix.lower() in ('.pt','.pth','.tmp'):
                        continue
                    if 'previews' in path.relative_to(run_root).parts: continue
                    archive.write(path, path.relative_to(run_root).as_posix())
            archive.write(lock, lock.relative_to(run_root).as_posix())
            archive.write(readme, readme.relative_to(run_root).as_posix())
        with zipfile.ZipFile(tmp) as archive:
            bad = archive.testzip()
            if bad: raise RuntimeError(f'결과 ZIP 검사 실패: {bad}')
        os.replace(tmp, result)
    status = {'status':'completed', 'gpu':gpu, 'summary':summary, 'result_zip':str(result),
              'classifier_training':False,
              'warning':'시각 검토용. 추후 성능 실험은 학습 구간만으로 GAN 재학습 또는 독립 새 출력 필요.'}
    (run_root/'run_status.json').write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding='utf-8')
    print('네 조건 생성 완료! 결과 ZIP:', result, flush=True)
except BaseException as exc:
    print('실행 오류:', traceback.format_exc(), flush=True)
    if run_root is not None:
        try:
            run_root.mkdir(parents=True, exist_ok=True)
            (run_root/'run_status.json').write_text(json.dumps({'status':'failed','error_type':type(exc).__name__,
                'error':str(exc),'traceback':traceback.format_exc()}, ensure_ascii=False, indent=2), encoding='utf-8')
        except Exception as save_exc:
            print('오류 기록 실패:', save_exc, flush=True)
finally:
    if mounted:
        try:
            print('Drive 동기화 중', flush=True)
            drive.flush_and_unmount()
        except BaseException as exc:
            print('Drive 동기화 확인 실패:', exc, flush=True)
    print('G4 런타임 반납 요청', flush=True)
    try: runtime.unassign()
    except BaseException as exc: print('자동 반납 확인 실패:', exc, flush=True)


## 결과

Drive `산공종설/results/gan_four_conditions_20261101_run01/`의 review 폴더에서 조건별 원본·생성 확인판과 결과 ZIP을 확인합니다. 원본 체크포인트는 Drive에 보관되며 ZIP에는 포함하지 않습니다.
